In [14]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout, BatchNormalization


In [15]:
FOLDER_PATH = "../img/v"
FRAME_COUNT = 16
IMG_SIZE = 64
CLASS_COUNT = 3 

In [16]:
class_names = ["0","1","2"]

In [17]:
def load_videos_from_new_folder(folder, frame_count=FRAME_COUNT, img_size=IMG_SIZE):
    videos = []
    labels = []

    for video_folder in os.listdir(folder):
        video_path = os.path.join(folder, video_folder)
        if os.path.isdir(video_path):
            try:
                video_id, label = video_folder.split("_")
                label = int(label)
            except ValueError:
                print(f"[SKIP] invalid dir name: {video_folder}")
                continue

            frames = []
            frame_files = sorted(os.listdir(video_path))[:frame_count]
            print(f"[INFO] {video_folder} -> {len(frame_files)} frame")

            for filename in frame_files:
                img_path = os.path.join(video_path, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    frames.append(img)
                else:
                    print(f"[WARN] frame can not be read!: {img_path}")

            if len(frames) == frame_count:
                video_array = np.array(frames).reshape(frame_count, img_size, img_size, 1)
                videos.append(video_array)
                labels.append(label)
            else:
                print(f"[SKIP] {video_folder} -> not enough frame ({len(frames)}/{frame_count})")

    print(f"[SUMMARY] Sum of videos: {len(videos)}")
    return np.array(videos), np.array(labels)


X, y = load_videos_from_new_folder(FOLDER_PATH, FRAME_COUNT, IMG_SIZE)
y = to_categorical(y, num_classes=CLASS_COUNT)

print("Number of videos:", len(X)) 
print("Shape of data:", X.shape) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


[INFO] 54_0 -> 16 frame
[INFO] 51_0 -> 16 frame
[INFO] 120_0 -> 16 frame
[INFO] 40_1 -> 16 frame
[INFO] 38_1 -> 16 frame
[INFO] 22_0 -> 16 frame
[INFO] 60_0 -> 16 frame
[INFO] 64_1 -> 16 frame
[INFO] 96_2 -> 16 frame
[INFO] 137_1 -> 16 frame
[INFO] 36_2 -> 16 frame
[INFO] 101_1 -> 16 frame
[INFO] 139_1 -> 16 frame
[INFO] 103_2 -> 16 frame
[INFO] 18_1 -> 16 frame
[INFO] 129_1 -> 16 frame
[INFO] 21_0 -> 16 frame
[INFO] 34_2 -> 16 frame
[INFO] 43_1 -> 16 frame
[INFO] 83_2 -> 16 frame
[INFO] 109_2 -> 16 frame
[INFO] 32_2 -> 16 frame
[INFO] 39_1 -> 16 frame
[INFO] 119_1 -> 16 frame
[INFO] 85_1 -> 16 frame
[INFO] 58_2 -> 16 frame
[INFO] 25_1 -> 16 frame
[INFO] 24_1 -> 16 frame
[INFO] 100_2 -> 16 frame
[INFO] 42_1 -> 16 frame
[INFO] 89_1 -> 16 frame
[INFO] 66_0 -> 16 frame
[INFO] 135_2 -> 16 frame
[INFO] 35_1 -> 16 frame
[INFO] 88_1 -> 16 frame
[INFO] 112_2 -> 16 frame
[INFO] 23_2 -> 16 frame
[INFO] 27_0 -> 16 frame
[INFO] 16_2 -> 16 frame
[INFO] 94_2 -> 16 frame
[INFO] 62_2 -> 16 frame
[INFO

In [18]:
print(f"Number of train sets {len(X_train)}")
print(f"Number of test sets {len(X_test)}")

Number of train sets 112
Number of test sets 29


In [20]:
input_shape = (FRAME_COUNT, IMG_SIZE, IMG_SIZE, 1)

model = Sequential([
    Conv3D(32, kernel_size=(3,3,3), activation='relu',padding="same", input_shape=input_shape),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Conv3D(64, kernel_size=(3,3,3), padding="same",activation='relu'),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Conv3D(64, kernel_size=(3,3,3),padding="same", activation='relu'),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Flatten(),

    Dense(128, activation='relu'),

    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X_train, y_train, epochs=10, batch_size=4, validation_split=0.1)

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.2f}")

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d_12 (Conv3D)              │ (None, 16, 64, 64, 32) │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_12 (MaxPooling3D) │ (None, 8, 32, 32, 32)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 8, 32, 32, 32)  │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_13 (Conv3D)              │ (None, 8, 32, 32, 64)  │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_13 (MaxPooling3D) │ (None, 4, 16, 16, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 4, 16, 16, 64)  │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_14 (Conv3D)              │ (None, 4, 16, 16, 64)  │       110,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_14 (MaxPooling3D) │ (None, 2, 8, 8, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 2, 8, 8, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,224,707 (4.67 MB)

 Trainable params: 1,224,387 (4.67 MB)

 Non-trainable params: 320 (1.25 KB)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 405ms/step - accuracy: 0.3325 - loss: 3.0371 - val_accuracy: 0.5833 - val_loss: 28.2287
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 354ms/step - accuracy: 0.4894 - loss: 2.0095 - val_accuracy: 0.5833 - val_loss: 5.7144
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 340ms/step - accuracy: 0.3914 - loss: 1.3526 - val_accuracy: 0.4167 - val_loss: 2.8045
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 378ms/step - accuracy: 0.3900 - loss: 1.3536 - val_accuracy: 0.3333 - val_loss: 2.0345
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 350ms/step - accuracy: 0.5312 - loss: 1.0176 - val_accuracy: 0.5833 - val_loss: 1.7891
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 359ms/step - accuracy: 0.5160 - loss: 1.1194 - val_accuracy: 0.5000 - val_loss: 1.2605
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 379ms/step - accuracy: 0.4252 - loss: 1.2971 - val_accuracy: 0.5000 - val_loss: 1.1041
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 326ms/step - accuracy: 0.6043 - loss: 0.8970 - val_accuracy: 